In [ ]:
# CHAMPION

# model_paths_ = {
#                 '/kaggle/input/pii-deberta-models/cuerpo-de-piiranha' : (1/3)/10,
#                 '/kaggle/input/pii-models/piidd-org-sakura' : (1/3)/10,
#                 '/kaggle/input/pii-detect-deberta3large-models/splendid-totem-15-checkpoint-1800-f1_0.9659' : (1/3)/10,
#                 '/kaggle/input/pii-deberta-models/cabeza-del-piinguuino': 7/10,
#                 '/kaggle/input/37vp4pjt': (1)/10,
#                 '/kaggle/input/pii-deberta-models/cabeza-de-piiranha': (0.5)/10,
#                 '/kaggle/input/pii-deberta-models/cabeza-de-piiranha-persuade_v0':(1/6)/10,
#                 '/kaggle/input/pii-deberta-models/cola del piinguuino' : (1/6)/10,
#                 '/kaggle/input/pii-deberta-models/cola-de-piiranha':(1/6)/10,
#                 }

# model_paths_ = {
#     '/kaggle/input/37vp4pjt': 5/10,
#     '/kaggle/input/pii-deberta-models/cuerpo-de-piiranha': 2/10,
#     '/kaggle/input/pii-deberta-models/cola del piinguuino' : 1/10,
#     '/kaggle/input/pii-deberta-models/cabeza-del-piinguuino': 10/10,
#     '/kaggle/input/pii-deberta-models/cabeza-de-piiranha': 3/10,
#     '/kaggle/input/pii-deberta-models/cola-de-piiranha':1/10,
#     '/kaggle/input/pii-models/piidd-org-sakura': 2/10,
#     '/kaggle/input/pii-deberta-models/cabeza-de-piiranha-persuade_v0':1/10,
#     }


# model_paths = {
#     '/kaggle/input/37vp4pjt': 10/10,
#     '/kaggle/input/pii-deberta-models/cuerpo-de-piiranha': 2/10,
#     '/kaggle/input/pii-deberta-models/cola del piinguuino' : 1/10,
#     '/kaggle/input/pii-deberta-models/cabeza-del-piinguuino': 5/10,
#     '/kaggle/input/pii-deberta-models/cabeza-de-piiranha': 3/10,
#     '/kaggle/input/pii-deberta-models/cola-de-piiranha':1/10,
#     '/kaggle/input/pii-models/piidd-org-sakura': 2/10,
#     '/kaggle/input/pii-deberta-models/cabeza-de-piiranha-persuade_v0':1/10,
#     }

In [ ]:
model_paths_ = {
                '/kaggle/input/pii-deberta-models/cabeza-del-piinguuino': 6.9/10,
    
                '/kaggle/input/37vp4pjt': (0.23)/10,
                
                '/kaggle/input/pii-deberta-models/cuerpo-de-piiranha' : (1.05)/10,
#                 '/kaggle/input/pii-models/piidd-org-sakura' : (0.16)/10,
                '/kaggle/input/pii-detect-deberta3large-models/splendid-totem-15-checkpoint-1800-f1_0.9659' : (0.2)/10,
                
                '/kaggle/input/pii-deberta-models/cabeza-de-piiranha': (0.8)/10,
                
                '/kaggle/input/pii-deberta-models/cabeza-de-piiranha-persuade_v0':(0.35)/10,
                '/kaggle/input/pii-deberta-models/cola del piinguuino' : (0.07)/10,
                '/kaggle/input/pii-deberta-models/cola-de-piiranha':(0.4)/10,
                }

# Installation

In [ ]:
# special version on gpu-onnxruntime for kaggle cuda12 environment
!pip install -q /kaggle/input/onyx-runtime-gpu-whl/onnxruntime_gpu-1.17.1-cp310-cp310-manylinux_2_28_x86_64.whl --force-reinstall --no-index --find-links="/kaggle/input/onyx-runtime-gpu-whl"
!pip install -q /kaggle/input/onyx-runtime-gpu-whl/onnx-1.16.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl --no-index --find-links="/kaggle/input/onyx-runtime-gpu-whl"
!pip install -q /kaggle/input/onyx-runtime-gpu-whl/onnxconverter_common-1.14.0-py2.py3-none-any.whl --no-index --find-links="/kaggle/input/onyx-runtime-gpu-whl"

In [ ]:
# Toggle to use training set folds for model
debug_on_train_df = False

# Enable to convert models for inference on-the-fly.
convert_before_inference = False

# Temporary directory for saving datasets and intermediate files.
temp_data_folder = "/tmp/output/"


In [ ]:
import os
import gc
from tqdm.auto import tqdm
import json
import numpy as np 
import pandas as pd 
from itertools import chain
from text_unidecode import unidecode
from typing import Dict, List, Tuple
import codecs
from datasets import Dataset, load_from_disk
from sklearn.metrics import log_loss
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import pickle
import re
from transformers import TrainingArguments, AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification
from scipy.special import softmax
from spacy.lang.en import English

# ⚙️ Config

In [ ]:
class config:
    device = 'cpu' 
    seed = 69
    # dataset path 
    train_dataset_path = "/kaggle/input/pii-detection-removal-from-educational-data/train.json"
    test_dataset_path = "/kaggle/input/pii-detection-removal-from-educational-data/test.json"
    sample_submission_path = "/home/nischay/PID/Data/sample_submission.csv"
       
    save_dir = temp_data_folder + "1/"

    #tokenizer params
    downsample = 0.45
    truncation = True 
    padding = False #'max_length'
    max_length = 3574
    doc_stride = 512
    
    target_cols = ['B-EMAIL', 'B-ID_NUM', 'B-NAME_STUDENT', 'B-PHONE_NUM', 
    'B-STREET_ADDRESS', 'B-URL_PERSONAL', 'B-USERNAME', 'I-ID_NUM', 
    'I-NAME_STUDENT', 'I-PHONE_NUM', 'I-STREET_ADDRESS', 'I-URL_PERSONAL','O']

    load_from_disk = None
    #training params
    learning_rate = 1e-5
    batch_size = 1
    epochs = 4
    NFOLDS = [0]
    trn_fold = 0
    # Initialize a tokenizer and model from the pretrained model path
    model_paths = model_paths_
    converted_path = '/kaggle/input/toonnx2-converted-models'


In [ ]:
if not os.path.exists(config.save_dir):
    os.makedirs(config.save_dir)

In [ ]:
nlp = English()
INFERENCE_MAX_LENGTH = 3500
threshold = 0.99
email_regex = re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+')
phone_num_regex = re.compile(r"(\(\d{3}\)\d{3}\-\d{4}\w*|\d{3}\.\d{3}\.\d{4})\s")
url_regex = re.compile(
    r'http[s]?://'  # http or https
    r'(?:(?:[A-Z0-9](?:[A-Z0-9-]{0,61}[A-Z0-9])?\.)+(?:[A-Z]{2,6}\.?|[A-Z0-9-]{2,}\.?)|'  # domain...
    r'localhost|'  # localhost...
    r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})'  # ...or ip
    r'(?::\d+)?'  # optional port
    r'(?:/?|[/?]\S+)', re.IGNORECASE)
street_regex = re.compile(r'\d{1,4} [\w\s]{1,20}(?:street|apt|st|avenue|ave|road|rd|highway|hwy|square|sq|trail|trl|drive|dr|court|ct|parkway|pkwy|circle|cir|boulevard|blvd)\W?(?=\s|$)', re.IGNORECASE)

# 📊 Preprocessing

In [ ]:
def find_span(target: list[str], document: list[str]) -> list[list[int]]:
    idx = 0
    spans = []
    span = []

    for i, token in enumerate(document):
        if token != target[idx]:
            idx = 0
            span = []
            continue
        span.append(i)
        idx += 1
        if idx == len(target):
            spans.append(span)
            span = []
            idx = 0
            continue
    
    return spans

In [ ]:
data = json.load(open(config.train_dataset_path))
test_data = json.load(open(config.test_dataset_path))

print('num_samples:', len(data))
print(data[0].keys())


In [ ]:
all_labels = sorted(list(set(chain(*[x["labels"] for x in data]))))
label2id = {l: i for i,l in enumerate(all_labels)}
id2label = {v:k for k,v in label2id.items()}

print(id2label)

In [ ]:
first_model_path = list(config.model_paths.keys())[0]
tokenizer = AutoTokenizer.from_pretrained(first_model_path)

In [ ]:
df_train = pd.DataFrame(data)
df_train.head(5)

In [ ]:
df_train['fold'] = df_train['document'] % 4
df_train.head(3)

In [ ]:
df_test = pd.DataFrame(test_data)
df_test.head(3)

In [ ]:
def downsample_df(train_df, percent):

    train_df['is_labels'] = train_df['labels'].apply(lambda labels: any(label != 'O' for label in labels))
    
    true_samples = train_df[train_df['is_labels'] == True]
    false_samples = train_df[train_df['is_labels'] == False]
    
    n_false_samples = int(len(false_samples) * percent)
    downsampled_false_samples = false_samples.sample(n=n_false_samples, random_state=42)
    
    downsampled_df = pd.concat([true_samples, downsampled_false_samples])    
    return downsampled_df

In [ ]:
def tokenize_row(example):
    text = []
    token_map = []
    
    idx = 0
    
    for t, ws in zip(example["tokens"], example["trailing_whitespace"]):
        text.append(t)
        token_map.extend([idx]*len(t))
        if ws:
            text.append(" ")
            token_map.append(-1)
            
        idx += 1
        
    tokenized = tokenizer("".join(text), return_offsets_mapping=True, truncation=config.truncation, max_length=config.max_length)
    
    return {
        "input_ids": tokenized.input_ids,
        "attention_mask": tokenized.attention_mask,
        "offset_mapping": tokenized.offset_mapping,
        "token_map": token_map,}

In [ ]:
df_test.describe()

In [ ]:
# %%time

if debug_on_train_df:


    if config.load_from_disk is None:
        
        df_train['fold'] = df_train['document'] % 4
        df_train.head(3)
        
        for i in range(-1, 4):    
            train_df = df_train[df_train['fold']==i].reset_index(drop=True)

            if i==config.trn_fold:
                config.valid_stride = True
            if i!=config.trn_fold and config.downsample > 0:
                train_df = downsample_df(train_df, config.downsample)
                config.valid_stride = False

            train_df = train_df
            print(len(train_df))
            ds = Dataset.from_pandas(train_df)

            ds = ds.map(
              tokenize_row,
              batched=False,
              num_proc=2,
              desc="Tokenizing",
            )

            ds.save_to_disk(f"{config.save_dir}fold_{i}.dataset")
            with open(f"{config.save_dir}_pkl", "wb") as fp:
                pickle.dump(train_df, fp)
            print("Saving dataset to disk:", config.save_dir)
        
else:
    
    if config.load_from_disk is None:

        config.valid_stride = True
        print(len(df_test))

        ds = Dataset.from_pandas(df_test)
        ds = ds.map(
          tokenize_row,
          batched=False,
          num_proc=2,
          desc="Tokenizing",
          )

        ds.save_to_disk(f"{config.save_dir}test.dataset")
        print("Saving dataset to disk:", config.save_dir)
      
        

In [ ]:
ds[0].keys()

# Inference & quantization

In [ ]:

def process_predictions(flattened_preds):
    """
    Processes each prediction in flattened predictions by applying softmax to convert logits to probabilities.

    Parameters:
    - flattened_preds: Iterable of prediction tensors.

    Returns:
    - List of predictions after applying softmax.
    """

    # Initialize a list to hold softmax-applied predictions
    predictions_softmax_all = []

    # Iterate over each set of predictions in the input
    for predictions in flattened_preds:
        # Apply softmax to convert logits to probabilities
        predictions_softmax = torch.softmax(predictions, dim=-1)
        # Append the softmax predictions to the result list
        predictions_softmax_all.append(predictions_softmax)

    # Return the list of predictions with softmax applied
    return predictions_softmax_all


In [ ]:
from transformers.convert_graph_to_onnx import convert
from onnxconverter_common import auto_convert_mixed_precision_model_path
import onnx
import torch.onnx
import onnxruntime

def predict_and_convert(data_loader, model, config, onnx_model_path):
    """
    Exports the given model to the ONNX format after processing a single batch from the data loader.

    Parameters:
    - data_loader: DataLoader object to provide input data for the model.
    - model: The model to be exported to ONNX format.
    - config: Configuration object containing device and others
    - onnx_model_path: Path where the ONNX model will be saved.

    Returns:
    - prediction_outputs: List of model outputs for the processed batch. Currently initialized but not used.
    """

    # Set the model to evaluation mode
    model.eval()

    # Initialize a list to store the prediction outputs
    prediction_outputs = []

    # Create an iterator from the DataLoader
    data_iter = iter(data_loader)

    # Fetch the first batch of data from the iterator
    batch = next(data_iter)

    # Disable gradient calculations for export
    with torch.no_grad():

        # Prepare inputs by reshaping and moving them to the specified device
        inputs = {key: val.reshape(val.shape[0], -1).to(config.device) for key, val in batch.items() if key in ['input_ids', 'attention_mask']}
        input_ids = inputs['input_ids']
        attention_mask = inputs['attention_mask']

        # Export the model to ONNX format with the specified configurations
        torch.onnx.export(model,  # Model to be exported
                          args=(input_ids, attention_mask),  # Example model input
                          f=onnx_model_path,  # Path to save the ONNX model
                          opset_version=12,  # ONNX opset version
                          input_names=['input_ids', 'attention_mask'],  # Names of the input parameters
                          output_names=['logits'],  # Names of the output
                          dynamic_axes={'input_ids': {0: 'batch_size', 1: 'sequence_length'},  # Dynamic axes for batching
                                        'attention_mask': {0: 'batch_size', 1: 'sequence_length'}}
                          )

    print("Model saved to", onnx_model_path)

    return prediction_outputs


def predict_and_quant(data_loader, config, original_onnx_model_path, output_file_name, data_path):
    """
    Performs quantization on a given ONNX model based on a single batch from the data loader and saves the quantized model.

    Parameters:
    - data_loader: DataLoader object providing input data for quantization.
    - config: Configuration object containing device settings.
    - original_onnx_model_path: Path to the original ONNX model that will be quantized.
    - output_file_name: Filename for the quantized ONNX model.
    - data_path: Path where additional data related to quantization might be stored.

    Returns:
    - prediction_outputs: List of model outputs for the processed batch. Currently, it only appends a placeholder value.
    """

    # Initialize a list to store prediction outputs
    prediction_outputs = []

    # Create an iterator from the DataLoader
    data_iter = iter(data_loader)

    # Fetch the first batch of data from the iterator
    batch = next(data_iter)

    # Disable gradient calculations for efficiency
    with torch.no_grad():
        
        # Prepare inputs by reshaping and moving them to the specified device
        inputs = {key: val.reshape(val.shape[0], -1).to(config.device) for key, val in batch.items() if key in ['input_ids', 'attention_mask']}
        
        input_ids = inputs['input_ids']
        attention_mask = inputs['attention_mask']

        # Quantization process
        print("Quantization")

        # Prepare input data for quantization by moving tensors to CPU and converting to numpy arrays
        input_data = {"input_ids": input_ids.cpu().numpy(), "attention_mask": attention_mask.cpu().numpy()}
            
        # Call the function to auto convert the ONNX model to mixed precision with specified settings
        auto_convert_mixed_precision_model_path(
            original_onnx_model_path,  # Original ONNX model path
            input_data,  # Input data for calibration during quantization
            output_file_name,  # Output file name for the quantized model
            provider=['CUDAExecutionProvider'],  # Specify the execution provider, can be changed to CPU if necessary
            location=data_path,  # specify the path to save external data tensors
            rtol=2,  # Relative tolerance for quantization
            atol=20,  # Absolute tolerance for quantization
            keep_io_types=True,  # Maintain input/output types
            verbose=True  # Enable verbose output during quantization
        )

        # Append a placeholder value to prediction outputs (currently not used for actual predictions)
        prediction_outputs.append(0)

    print("Model saved to", output_file_name)

    return prediction_outputs
                        

def predict(data_loader, session, config):
    """
    Performs inference using a given ONNX model session over all batches from a data loader.

    Parameters:
    - data_loader: DataLoader object providing batches of input data for inference.
    - session: The ONNX runtime session initialized with the model to be used for inference.
    - config: Configuration object containing settings

    Returns:
    - processed_predictions: List of processed predictions after inference for all input data.
    """

    # Initialize a list to collect raw predictions for each batch
    prediction_outputs = []

    # Iterate over all batches of data from the data loader
    for batch in tqdm(data_loader, desc="Predicting"):
        with torch.no_grad():
            # Prepare inputs by reshaping and moving them to the specified device
            inputs = {key: val.reshape(val.shape[0], -1).to(config.device) for key, val in batch.items() if key in ['input_ids', 'attention_mask']}
            
            # Retrieve the names of the input and output nodes from the model session
            input_names = [inp.name for inp in session.get_inputs()]
            output_names = [out.name for out in session.get_outputs()]
            
            # Extract input_ids and attention_mask from inputs
            input_ids = inputs['input_ids']
            attention_mask = inputs['attention_mask']

            # Prepare input data by moving tensors to CPU and converting to numpy arrays
            input_data = {"input_ids": input_ids.cpu().numpy(), "attention_mask": attention_mask.cpu().numpy()}

            # Execute the model
            onnx_outputs = session.run(None, input_data)

            # Append raw model outputs (predictions) to the list
            prediction_outputs.append(torch.tensor(onnx_outputs[0]))  # Assuming the first output is what we need

    # Flatten the list of predictions across all batches
    prediction_outputs = [logit for batch in prediction_outputs for logit in batch]

    # Process the predictions as required (e.g., applying softmax, thresholding)
    processed_predictions = process_predictions(prediction_outputs)

    return processed_predictions


In [ ]:
def process_predictions_ans(flattened_preds, threshold=0.95):
    """
    Processes predictions by applying a threshold to distinguish between a specific class and others.
    It assumes softmax has already been applied to the predictions.

    Parameters:
    - flattened_preds: A list of prediction tensors, with each tensor representing predictions for a batch.
    - threshold: A probability threshold used to decide whether to classify a prediction as a specific class or as 'other'.

    Returns:
    - preds_final: A list of numpy arrays with final predictions after applying the threshold.
    """
    
    print("\nPrediction")
    preds_final = []  # Initialize a list to store final predictions

    # Iterate over each set of predictions
    for predictions in flattened_preds:
        # softmax was applied to the first dimension before averaging
        predictions_softmax = predictions

        # Get the argmax across all classes
        predictions_argmax = predictions.argmax(-1)

        # Get predictions for all classes except 'O'
        predictions_without_O = predictions_softmax[:, :12].argmax(-1)

        # Get the softmax probabilities for the 'O' class
        O_predictions = predictions_softmax[:, 12]

        # Apply threshold to decide between 'O' class and other classes
        pred_final = torch.where(O_predictions < threshold, predictions_without_O, predictions_argmax)

        # Convert final predictions to numpy array and add to the list
        preds_final.append(pred_final.numpy())

    return preds_final


In [ ]:
# Load Tokenized dataset from disk
keep_cols = {"input_ids", "attention_mask"}
collator = DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=512)

if not debug_on_train_df:
    test_ds = load_from_disk(f'{config.save_dir}test.dataset')
    test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep_cols])
    config.data_length = len(test_ds)
    config.len_token = len(tokenizer)
    print('Dataset Loaded....')
    print((test_ds[0].keys()))
    print("Generating Test DataLoader")
    test_dataloader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False, num_workers=4, pin_memory=False, collate_fn=collator)

else:
    # Fallback for memory tests of several models on fold zero
    fold = config.trn_fold
    test_ds = load_from_disk(f'{config.save_dir}fold_{fold}.dataset')
    test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep_cols])
    config.data_length = len(test_ds)
    config.len_token = len(tokenizer)
    print('Dataset Loaded....')
    print(test_ds)
    print((test_ds[0].keys()))
    print("Generating Test DataLoader")
    test_dataloader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False, num_workers=4, pin_memory=False, collate_fn=collator)

In [ ]:
# All predict data
predictions_softmax_logits = []
all_preds = []

for model_path, weight in config.model_paths.items():
    
    fold = config.trn_fold
    
#     if convert_before_inference:
    if 'pii-detect-deberta3large-models' in model_path: 
        
        quantized_model_name = "/kaggle/input/pii-detect-onnx-models" + "/optimized" + model_path.split("/")[-1] + "_f" + str(fold) + ".onnx"

#         # Loading the original model and converting it to ONNX
#         model = AutoModelForTokenClassification.from_pretrained(model_path)

#         # Converting it to ONNX to a temp folder
#         converted_model_name = temp_data_folder + "original_model.onnx"
#         predictions_softmax_all = predict_and_convert(test_dataloader, model, config, converted_model_name)
#         del model
#         gc.collect()
#         torch.cuda.empty_cache()

#         # In commit mode, save all quantized models with different names to create a dataset and reuse them later bypassing
#         #vquantization and conversion
#         quantized_model_name = "/kaggle/working/optimized" + model_path.split("/")[-1] + "_f" + str(fold) + ".onnx"
#         # data path should be relative
#         quantized_data_path = "optimized" + model_path.split("/")[-1] + "_f" + str(fold) + ".data"
        
#         # Quantization
#         predictions_softmax_all = predict_and_quant(test_dataloader, config, converted_model_name, quantized_model_name, quantized_data_path)
    
    else:
        # Use already converted models, you can make a commit notebook once and save output models to a dataset,
        # for example, /kaggle/input/toonnx2-converted-models    
        quantized_model_name = config.converted_path + "/optimized" + model_path.split("/")[-1] + "_f" + str(fold) + ".onnx"

    
    # Inference with ONNX
    print(f"Inference: {model_path}")
    
    # Create ONNX Runtime session for GPU
    session = onnxruntime.InferenceSession(quantized_model_name, providers=['CUDAExecutionProvider'])
    # Uncomment this if you want to debug something on CPU
    # session = onnxruntime.InferenceSession(quantized_model_name)
    
    # Predict 
    predictions_softmax_all = predict(test_dataloader, session, config)
    
    # Keep all logits for ensemble later
    predictions_softmax_logits.append(predictions_softmax_all)
    
del test_dataloader, test_ds
gc.collect()
torch.cuda.empty_cache()


In [ ]:
print("onnx",onnxruntime.__version__)
!nvcc --version

# Making final answer

In [ ]:
# Initialize an empty list to store the mean of the softmax predictions from all models.
predictions_mean_all = []

# Calculate the total weight of all models to normalize the weights if its sum exceeds 1.
total_weight = sum(config.model_paths.values())
print(f"Total weight: {total_weight}")

# Retrieve the individual weights for each model.
model_weights = list(config.model_paths.values())

# Iterate over each sample since the length of texts can vary.
for sample_index in range(len(predictions_softmax_logits[0])):
    
    # Initialize a tensor to accumulate weighted predictions for the current sample.
    weighted_predictions_sum = torch.zeros(predictions_softmax_logits[0][sample_index].size())

    # Iterate over each model to compute its contribution to the final prediction.
    for model_index in range(len(predictions_softmax_logits)):
        weighted_prediction = predictions_softmax_logits[model_index][sample_index] * (model_weights[model_index] / total_weight)
        weighted_predictions_sum += weighted_prediction

    # Append the mean of the weighted predictions for the current sample to the list.
    predictions_mean_all.append(weighted_predictions_sum)



In [ ]:
processed_predictions = process_predictions_ans(predictions_mean_all)
print(len(processed_predictions), processed_predictions[0].shape)

In [ ]:
triplets = []
pairs = set()  # membership operation using set is faster O(1) than that of list O(n)
processed = []
emails = []
phone_nums = []
urls = []
streets = []
print(id2label)

# For each prediction, token mapping, offsets, tokens, and document in the dataset
for p, token_map, offsets, tokens, doc, full_text in zip(
    processed_predictions, 
    ds["token_map"], 
    ds["offset_mapping"], 
    ds["tokens"], 
    ds["document"],
    ds["full_text"]
):

    # Iterate through each token prediction and its corresponding offsets
    for token_pred, (start_idx, end_idx) in zip(p, offsets):
        label_pred = id2label[token_pred]  # Predicted label from token
        if start_idx + end_idx == 0:
            continue
        if token_map[start_idx] == -1:
            start_idx += 1
        while start_idx < len(token_map) and tokens[token_map[start_idx]].isspace():
            start_idx += 1
        if start_idx >= len(token_map):
            break
        token_id = token_map[start_idx]  # Token ID at start index
        if label_pred in ("O", "B-EMAIL", "B-PHONE_NUM", "I-PHONE_NUM") or token_id == -1:
            continue
        pair = (doc, token_id)
        if pair not in pairs:
            processed.append({"document": doc, "token": token_id, "label": label_pred, "token_str": tokens[token_id]})
            pairs.add(pair)
    
    # email
    for token_idx, token in enumerate(tokens):
        if re.fullmatch(email_regex, token) is not None:
            emails.append(
                {"document": doc, "token": token_idx, "label": "B-EMAIL", "token_str": token}
            )
                
    # phone number
    matches = phone_num_regex.findall(full_text)
    if not matches:
        continue
    for match in matches:
        target = [t.text for t in nlp.tokenizer(match)]
        matched_spans = find_span(target, tokens)
    for matched_span in matched_spans:
        for intermediate, token_idx in enumerate(matched_span):
            prefix = "I" if intermediate else "B"
            phone_nums.append(
                {"document": doc, "token": token_idx, "label": f"{prefix}-PHONE_NUM", "token_str": tokens[token_idx]}
            )
    
    # url
    matches = url_regex.findall(full_text)
    if not matches:
        continue
    for match in matches:
        target = [t.text for t in nlp.tokenizer(match)]
        matched_spans = find_span(target, tokens)
    for matched_span in matched_spans:
        for intermediate, token_idx in enumerate(matched_span):
            prefix = "I" if intermediate else "B"
            urls.append(
                {"document": doc, "token": token_idx, "label": f"{prefix}-URL_PERSONAL", "token_str": tokens[token_idx]}
            )
    

In [ ]:
df = pd.DataFrame(processed + phone_nums + emails + urls)

# Assign each row a unique 'row_id'
df["row_id"] = list(range(len(df)))

# Cast your findings into a CSV file for further exploration
df[["row_id", "document", "token", "label"]].to_csv("submission.csv", index=False)
df